In [ ]:
# Cell 1 — Load config
%run /home/jovyan/work/setup/config.py
import sys; sys.path.insert(0, "/home/jovyan/work")
from utils.delta_utils import save_layer

In [ ]:
# Cell 2 — Build dim_product (distinct brand/package combinations)
from pyspark.sql.functions import xxhash64, col

df_silver = spark.read.format("delta").load(f"{SILVER_PATH}/silver_beverage_sales_enriched")

dim_product = (
    df_silver
    .select("ce_brand_flvr", "brand_nm", "pkg_cat", "pkg_cat_desc", "tsr_pckg_nm")
    .dropDuplicates(["ce_brand_flvr", "pkg_cat", "tsr_pckg_nm"])
    .withColumn("product_sk", xxhash64(col("ce_brand_flvr"), col("pkg_cat"), col("tsr_pckg_nm")))
    .select("product_sk", "ce_brand_flvr", "brand_nm", "pkg_cat", "pkg_cat_desc", "tsr_pckg_nm")
)

save_layer(dim_product, "dim_product", GOLD_PATH, PG_WRITE_PROPS)
dim_product.show()